<a href="https://colab.research.google.com/github/flango2023/FarmTech-Solutions-Fase5/blob/main/RichardSchmitz_rm567951_pbl_fase5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FarmTech Solutions - Fase 5: Machine Learning em Produção Agrícola

**Desenvolvedor:** Richard Schmitz  
**RM:** 567951  
**Instituição:** FIAP  
**Data:** Março 2026

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print('Importações concluídas com sucesso!')

Importações concluídas com sucesso!


In [2]:
# Carregamento e Análise Exploratória de Dados (EDA)

# Carregar o dataset
df = pd.read_csv('crop_yield.csv')
print("Dataset carregado com sucesso!")
print(f"Dimensões: {df.shape}")
print("\nPrimeiras 5 linhas:")
print(df.head())

# Verificar tipos de dados e valores ausentes
print("\nInformações do dataset:")
print(df.info())

print("\nEstatísticas descritivas:")
print(df.describe())

# Verificar valores ausentes
print("\nValores ausentes por coluna:")
print(df.isnull().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'crop_yield.csv'

In [ ]:
# Visualizações EDA

# Histogramas das variáveis numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].hist(figsize=(12, 8), bins=20)
plt.suptitle('Distribuição das Variáveis Numéricas')
plt.tight_layout()
plt.show()

# Box plots para detectar outliers
plt.figure(figsize=(12, 6))
for i, col in enumerate(numeric_cols):
    plt.subplot(2, 3, i+1)
    sns.boxplot(y=df[col])
    plt.title(f'Box Plot - {col}')
plt.tight_layout()
plt.show()

# Matriz de correlação
plt.figure(figsize=(10, 8))
correlation_matrix = df[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Matriz de Correlação')
plt.show()

# Scatter plots das variáveis preditoras vs target
preditoras = [col for col in numeric_cols if col != 'Yield']
plt.figure(figsize=(15, 10))
for i, col in enumerate(preditoras):
    plt.subplot(2, 2, i+1)
    plt.scatter(df[col], df['Yield'], alpha=0.6)
    plt.xlabel(col)
    plt.ylabel('Yield')
    plt.title(f'{col} vs Yield')
plt.tight_layout()
plt.show()

In [ ]:
# Clustering com K-Means

# Preparação dos dados para clustering
features_clustering = preditoras
X_clustering = df[features_clustering]

# Normalização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clustering)

# Método do cotovelo para determinar K
inertia = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia, 'bx-')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia')
plt.title('Método do Cotovelo')
plt.grid(True)
plt.show()

# Aplicar K-Means com K=3 (baseado no cotovelo)
k_optimo = 3
kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualizar clusters (usando 2 primeiras componentes principais)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['Cluster'], cmap='viridis', alpha=0.7)
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('Clusters Visualizados com PCA')
plt.colorbar(scatter)
plt.show()

# Estatísticas por cluster
print("Estatísticas por Cluster:")
cluster_stats = df.groupby('Cluster')[numeric_cols].mean()
print(cluster_stats)

In [ ]:
# Detecção de Outliers

# Método IQR (Interquartile Range)
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Detectar outliers para cada variável numérica
outliers_summary = {}
for col in numeric_cols:
    outliers, lb, ub = detect_outliers_iqr(df, col)
    outliers_summary[col] = {
        'count': len(outliers),
        'percentage': len(outliers) / len(df) * 100,
        'lower_bound': lb,
        'upper_bound': ub
    }
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)")

# Visualizar outliers
plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_cols):
    plt.subplot(2, 3, i+1)
    outliers, lb, ub = detect_outliers_iqr(df, col)
    plt.scatter(df.index, df[col], alpha=0.6, label='Normal')
    if len(outliers) > 0:
        plt.scatter(outliers.index, outliers[col], color='red', label='Outlier', s=50)
    plt.axhline(y=lb, color='orange', linestyle='--', alpha=0.7, label='Limites IQR')
    plt.axhline(y=ub, color='orange', linestyle='--', alpha=0.7)
    plt.xlabel('Índice')
    plt.ylabel(col)
    plt.title(f'Outliers - {col}')
    plt.legend()
plt.tight_layout()
plt.show()

# Remover outliers para modelagem (opcional)
df_clean = df.copy()
for col in numeric_cols:
    outliers, lb, ub = detect_outliers_iqr(df, col)
    df_clean = df_clean[~df_clean.index.isin(outliers.index)]

print(f"\nDataset original: {df.shape[0]} amostras")
print(f"Dataset sem outliers: {df_clean.shape[0]} amostras")
print(f"Removidas: {df.shape[0] - df_clean.shape[0]} amostras")

In [ ]:
# Modelos Preditivos

# Preparação dos dados para modelagem
X = df_clean[preditoras]
y = df_clean['Yield']

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalização para modelos que precisam
scaler_model = StandardScaler()
X_train_scaled = scaler_model.fit_transform(X_train)
X_test_scaled = scaler_model.transform(X_test)

# Dicionário para armazenar resultados
resultados = {}

# Modelo 1: Regressão Linear
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
resultados['Linear Regression'] = {
    'MSE': mean_squared_error(y_test, y_pred_lr),
    'MAE': mean_absolute_error(y_test, y_pred_lr),
    'R2': r2_score(y_test, y_pred_lr)
}

# Modelo 2: Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)
resultados['Ridge Regression'] = {
    'MSE': mean_squared_error(y_test, y_pred_ridge),
    'MAE': mean_absolute_error(y_test, y_pred_ridge),
    'R2': r2_score(y_test, y_pred_ridge)
}

# Modelo 3: Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)  # Random Forest não precisa de scaling
y_pred_rf = rf.predict(X_test)
resultados['Random Forest'] = {
    'MSE': mean_squared_error(y_test, y_pred_rf),
    'MAE': mean_absolute_error(y_test, y_pred_rf),
    'R2': r2_score(y_test, y_pred_rf)
}

# Modelo 4: Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
resultados['Gradient Boosting'] = {
    'MSE': mean_squared_error(y_test, y_pred_gb),
    'MAE': mean_absolute_error(y_test, y_pred_gb),
    'R2': r2_score(y_test, y_pred_gb)
}

# Modelo 5: Support Vector Regression
svr = SVR(kernel='rbf', C=1.0)
svr.fit(X_train_scaled, y_train)
y_pred_svr = svr.predict(X_test_scaled)
resultados['SVR'] = {
    'MSE': mean_squared_error(y_test, y_pred_svr),
    'MAE': mean_absolute_error(y_test, y_pred_svr),
    'R2': r2_score(y_test, y_pred_svr)
}

# Comparação dos modelos
df_resultados = pd.DataFrame(resultados).T
print("Comparação dos Modelos:")
print(df_resultados)

# Visualizar R² dos modelos
plt.figure(figsize=(10, 6))
bars = plt.bar(df_resultados.index, df_resultados['R2'], color=['blue', 'green', 'red', 'purple', 'orange'])
plt.xlabel('Modelo')
plt.ylabel('R² Score')
plt.title('Comparação do R² dos Modelos')
plt.ylim(0, 1)
for bar, r2 in zip(bars, df_resultados['R2']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{r2:.3f}', ha='center', va='bottom')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.show()

# Melhor modelo
best_model = df_resultados['R2'].idxmax()
print(f"\nMelhor modelo: {best_model} com R² = {df_resultados.loc[best_model, 'R2']:.3f}")